# spi_nb_bronze_energy

**Purpose.** Bronze ingestion of the REE (Red Eléctrica de España) demand
evolution API into the Lakehouse, for the Energy source (source 3 of
`spi_pl_bronze`, per ADR-007).

**Inputs.** REE REST API
(`https://apidatos.ree.es/en/datos/demanda/evolucion`), parameterized by
`START_YEAR` and `END_YEAR`; three regions (`Espana`, `Madrid`, `Catalunya`).

**Outputs.** Delta table `spi_bronze_energy_raw` in the default Lakehouse,
written with `mode="overwrite"`. Every row carries `_ingestion_timestamp`
and `_source_api`; all columns are stored as string, per the Bronze layer
principle.

**Dependencies.** `pandas`, `requests`, `deltalake`. Runs on the Fabric
**Python** runtime (no Spark session), per ADR-012.

**Exit contract.** On success, calls `notebookutils.notebook.exit()` with
`{"rows_written": <int>}`. On failure, the exception propagates uncaught; a
chained Script activity reads the Notebook activity's status, per the
2026-09-30 addendum to ADR-009.

**Last updated.** 2026-09-30.

In [ ]:
from datetime import datetime, timezone
import json
import logging
import time

import pandas as pd
import requests
from deltalake import write_deltalake

# ────────────────────────────────────────────────────────────────────────────
# Configure environment variables for the API
# ────────────────────────────────────────────────────────────────────────────

URL_BASE = "https://apidatos.ree.es"
ENDPOINT = "/en/datos/demanda/evolucion"
TIME_TRUNC = "month"
GEO_TRUNC = "electric_system"
REGIONS = [
    {"geo_name": "Espana", "geo_id": "8741", "geo_limit": "peninsular"},
    {"geo_name": "Madrid", "geo_id": "13", "geo_limit": "ccaa"},
    {"geo_name": "Catalunya", "geo_id": "9", "geo_limit": "ccaa"},
]

TIMEOUT = 30
BASE = 2
MAX_ATTEMPTS = 4  # attempts = 1 + 3 retries

In [ ]:
START_YEAR = 2014
END_YEAR = datetime.now().year

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Configure Logging
# ────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("api_ree_bronze_load")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function build_request_url
# ────────────────────────────────────────────────────────────────────────────

def build_request_url(geo_limit: str, geo_ids: str, year: int) -> str:
    """
    Build the REE API URL with the constants and specified parameters.
    Args:
        geo_limit (str): The value of the geo_limit parameter for the URL.
        geo_ids (str): The value of the geo_ids parameter for the URL.   
    Returns:
        str: The complete URL to make the REE API call.
    """
    url = (
        f"{URL_BASE}{ENDPOINT}?"
        f"start_date={year}-01-01T00:00&"
        f"end_date={year}-12-31T23:59&"
        f"time_trunc={TIME_TRUNC}&"
        f"geo_trunc={GEO_TRUNC}&"
        f"geo_limit={geo_limit}&"
        f"geo_ids={geo_ids}"
    )
    return url

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function fetch_with_retry
# ────────────────────────────────────────────────────────────────────────────

def fetch_with_retry(url: str) -> dict:
    """ 
    Makes an API call with retries on failure.
    Args:
        url (str): The URL of the API to which the call will be made.
    Returns:
        dict: The API response in JSON format if the call is successful.
    Raises:
        Exception: If the maximum number of unsuccessful retries is reached.
    """
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            log.info(f"Attempt {attempt} of {MAX_ATTEMPTS}: calling the API...")
            response = requests.get(url, timeout=TIMEOUT)
            status = response.status_code

            #Success: 200-299
            if 200 <= status < 300:
                log.info("Successful API call.")
                return response.json()   
            
            #Temporary errors
            elif status in (429, 500, 502, 503, 504):
                if attempt < MAX_ATTEMPTS:
                    delay = BASE ** attempt
                    log.warning(f"Temporary error {status}. Retrying...")
                    time.sleep(delay)
                else:
                    log.error(f"Maximum number of retries reached. Persistent temporary error: {status}.")
                    raise Exception(f"Persistent temporary error in the API: {status}")
            
            #Permanent errors
            else:
                log.error(f"Unexpected error {status}. No retry.")
                raise Exception(f"Unexpected error in the API: {status}")
        
        except (requests.exceptions.RequestException):
            if attempt < MAX_ATTEMPTS:
                delay = BASE ** attempt
                log.info(f"Retrying in {delay} seconds...")
                time.sleep(delay)
            
            else:
                log.error("Maximum retries reached. API call failed.")
                raise

    raise RuntimeError("Error calling the API after several attempts.")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function parse_response
# ────────────────────────────────────────────────────────────────────────────

def parse_response(response: dict, geo_name: str) -> list:
    """
    Parse the API response and extract the relevant data.
    Args:
        response (dict): The API response in JSON format.
        geo_name (str): The name of the geolocation.
    Returns:
        list: A list of dictionaries with the parsed data.
    """
    log.info("Parsing API response.")
    rows = []
    for series in response["included"]:  
        series_type = series["type"] 
        for point in series["attributes"]["values"]:
            row = {
                "series_type": series_type,
                "geo_name": geo_name,
                "datetime": point.get("datetime"),
                "value": point.get("value"),
                "percentage": point.get("percentage")
            }
            rows.append(row)
    return rows

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function build_dataframe
# ────────────────────────────────────────────────────────────────────────────

def build_dataframe(data: list) -> pd.DataFrame:
    """
    Builds a pandas DataFrame from a list of dictionaries.
    Args:
        data (list): A list of dictionaries with the parsed data.
    Returns:
        pd.DataFrame: A DataFrame with the data.
    """
    log.debug("Building DataFrame.")
    df = pd.DataFrame(data)
    df["_ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
    df["_source_api"] = ENDPOINT
    df = df.astype(str) # Converted to String by the principle of bronze layer 
    return df

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create main to run the entire process
# ────────────────────────────────────────────────────────────────────────────

def main() -> int:
    """
    Main function that executes the entire process Data loading from the REE API.
    Returns:
        int: Number of rows written to the Bronze table.
    """
    all_data = []
    for region in REGIONS:
        geo_name = region["geo_name"]
        geo_id = region["geo_id"]
        geo_limit = region["geo_limit"]
        for year in range(START_YEAR, END_YEAR + 1):
            url = build_request_url(geo_limit, geo_id, year)
            log.info(f"Processing region: {geo_name}/{year}")

            response = fetch_with_retry(url)
            parsed_data = parse_response(response, geo_name)
            all_data.extend(parsed_data)
            year_rows = len(parsed_data)
            total_rows = len(all_data)
            log.info(f"{geo_name} ({year}): {year_rows} rows/total: {total_rows}")

    df = build_dataframe(all_data)
    table_uri = (
        "abfss://bbffcd8e-0f61-436d-bee8-eac2d26d38b2@onelake.dfs.fabric.microsoft.com/"
        "c6048c3c-087e-4436-80fa-6013d1a04aeb/Tables/spi_bronze_energy_raw"
    )
    storage_options = {
        "bearer_token": notebookutils.credentials.getToken("storage"),
        "use_fabric_endpoint": "true",
    }
    write_deltalake(table_uri, df, mode="overwrite", storage_options=storage_options)
    log.info(f"Data stored in spi_bronze_energy_raw: {len(df)} rows")
    return len(df)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Exit contract — outside any try/except (ADR-009, 2026-09-30 addendum)
# ────────────────────────────────────────────────────────────────────────────

rows_written = main()
notebookutils.notebook.exit(json.dumps({"rows_written": rows_written}))